# Mixture-of-Experts Comparison on Kestrel

Does training separate XGBoost models per wallclock cluster improve runtime
prediction compared to a single XGBoost model on all the data?

**Comparison:**
- **Single model:** one XGBoost trained on all jobs (current approach)
- **Per-bin experts:** separate XGBoost per wallclock cluster, predictions combined

**Dataset:** NLR Kestrel, benchmark window 2025-03-29 to 2025-06-26

**Related:** Issue [#124](https://github.com/NatLabRockies/hpc-oda-commons/issues/124)

## 1. Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from hpc_oda_commons.models.job_runtime_xgboost.model import (
    JobRuntimeXGBoostConfig,
    JobRuntimeXGBoostModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

In [ ]:
# Load and slice to benchmark window
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 3, 29, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()

# Use a sample to keep runtime manageable
SAMPLE_SIZE = 30_000
full_window_size = len(df)
if len(df) > SAMPLE_SIZE:
    df = df.tail(SAMPLE_SIZE).copy()

rows_all = df.to_dict('records')
print(f'Full window: {full_window_size:,} rows')
print(f'Using: {len(rows_all):,} rows (last {SAMPLE_SIZE:,} from window)')

## 2. Define wallclock bins

Based on the detected partition limits from the wallclock distribution analysis.

In [ ]:
BIN_EDGES_H = [0, 1, 2, 4, 12, 24, 48, float('inf')]
BIN_LABELS  = ['<=1h', '1-2h', '2-4h', '4-12h', '12-24h', '24-48h', '>48h']
BIN_EDGES_S = [e * 3600 for e in BIN_EDGES_H]

def assign_bin(wc_seconds):
    for i in range(len(BIN_EDGES_S) - 1):
        if wc_seconds <= BIN_EDGES_S[i + 1]:
            return BIN_LABELS[i]
    return BIN_LABELS[-1]

df['wc_bin'] = df['requested_seconds'].apply(assign_bin)

print(f'{"Bin":<10} {"Jobs":>8} {"Pct":>6} {"RT median":>10} {"RT mean":>10}')
print('-' * 50)
for label in BIN_LABELS:
    subset = df[df['wc_bin'] == label]
    if len(subset) == 0:
        continue
    rt = subset['runtime_seconds'].dropna()
    print(f'{label:<10} {len(subset):>8,} {len(subset)/len(df)*100:>5.1f}% '
          f'{rt.median():>10,.0f}s {rt.mean():>10,.0f}s')

## 3. Single Model (current approach)

One XGBoost trained on all jobs. This is the current default — the model sees
`requested_seconds` as a feature and can split on it internally.

In [ ]:
CONFIG = dict(
    n_windows=4,
    test_window_hours=6,
    training_lookback_days=30,
    max_svd_components=16,
    target_max_one_hot_width=128,
    random_state=42,
)

print('Running single model on all data...')
model_single = JobRuntimeXGBoostModel(JobRuntimeXGBoostConfig(**CONFIG))
payload_single = model_single.evaluate(rows_all)

single_mae  = payload_single['mae']
single_rmse = payload_single['rmse']
single_scored = payload_single['summary']['rows_scored']

print(f'  MAE:    {single_mae:,.1f}s')
print(f'  RMSE:   {single_rmse:,.1f}s')
print(f'  Scored: {single_scored:,} rows')

## 4. Per-Bin Experts (mixture-of-experts approach)

Split the data into wallclock bins. Train a separate XGBoost per bin.
Each expert only sees jobs from its cluster.

In [ ]:
print('Running per-bin experts...\n')

# Group rows by bin
bin_rows = {label: [] for label in BIN_LABELS}
for row in rows_all:
    wc = row.get('requested_seconds')
    if wc is None or wc <= 0:
        continue
    bin_rows[assign_bin(wc)].append(row)

expert_results = {}
expert_y_true = []
expert_y_pred = []

for label in BIN_LABELS:
    rows_bin = bin_rows[label]
    if len(rows_bin) < 50:
        print(f'  {label}: {len(rows_bin)} rows — skipping (too few)')
        continue
    
    try:
        model = JobRuntimeXGBoostModel(JobRuntimeXGBoostConfig(**CONFIG))
        payload = model.evaluate(rows_bin, capture_artifacts=True)
        scored = payload['summary']['rows_scored']
        
        if scored == 0:
            print(f'  {label}: {len(rows_bin):,} rows — 0 scored (no valid rolling windows)')
            continue
        
        expert_results[label] = payload
        print(f'  {label}: {len(rows_bin):,} rows, scored={scored:,}, '
              f'MAE={payload["mae"]:,.1f}s, RMSE={payload["rmse"]:,.1f}s')
        
        # Collect raw predictions for correct global metric aggregation
        if '_y_true' in payload and '_y_pred' in payload:
            expert_y_true.extend(payload['_y_true'])
            expert_y_pred.extend(payload['_y_pred'])
    except ValueError as e:
        print(f'  {label}: {len(rows_bin):,} rows — failed: {e}')

# Combined metrics from all experts (correct aggregation)
if expert_y_true:
    expert_mae  = np.mean(np.abs(np.array(expert_y_true) - np.array(expert_y_pred)))
    expert_rmse = np.sqrt(np.mean((np.array(expert_y_true) - np.array(expert_y_pred))**2))
    expert_scored = len(expert_y_true)
    print(f'\n  COMBINED: MAE={expert_mae:,.1f}s, RMSE={expert_rmse:,.1f}s, scored={expert_scored:,}')
else:
    # Fallback: weighted MAE from per-expert results
    total = sum(r['summary']['rows_scored'] for r in expert_results.values())
    expert_mae = sum(r['mae'] * r['summary']['rows_scored'] for r in expert_results.values()) / total if total else 0
    expert_rmse = None
    expert_scored = total
    print(f'\n  COMBINED (weighted MAE): MAE={expert_mae:,.1f}s, scored={expert_scored:,}')
    print('  Note: could not aggregate RMSE — capture_artifacts may not have returned predictions')

## 5. Results

In [ ]:
print('=' * 60)
print('RESULTS: SINGLE MODEL vs PER-BIN EXPERTS')
print('=' * 60)

print(f'\n{"Approach":<35} {"MAE":>10} {"RMSE":>10} {"Scored":>8}')
print('-' * 70)
print(f'{"Single model (current)":<35} {single_mae:>10,.1f}s {single_rmse:>10,.1f}s {single_scored:>8,}')

if expert_rmse is not None:
    print(f'{"Per-bin experts (combined)":<35} {expert_mae:>10,.1f}s {expert_rmse:>10,.1f}s {expert_scored:>8,}')
else:
    print(f'{"Per-bin experts (combined)":<35} {expert_mae:>10,.1f}s {"N/A":>10} {expert_scored:>8,}')

mae_diff = (expert_mae - single_mae) / single_mae * 100
print(f'\nMAE difference: {mae_diff:+.1f}%')

if mae_diff < -5:
    print('Per-bin experts IMPROVE over single model — mixture approach has value.')
elif mae_diff > 5:
    print('Single model is BETTER — experts hurt due to less training data per bin.')
    print('XGBoost already captures the wallclock signal through its internal tree splits.')
else:
    print('Results are SIMILAR — the extra complexity of per-bin experts may not be justified.')

In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(7, 5))

approaches = ['Single model\n(current)', 'Per-bin experts\n(mixture)']
maes = [single_mae, expert_mae]
colors = ['steelblue', 'seagreen']

bars = ax.bar(approaches, maes, color=colors, width=0.5)
for bar, mae in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{mae:,.0f}s', ha='center', fontsize=14, fontweight='bold')

ax.set_ylabel('MAE (seconds, lower = better)')
ax.set_title(f'Single Model vs Per-Bin Experts\nNLR Kestrel ({expert_scored + single_scored:,} scored rows)')
plt.tight_layout()
plt.show()

## 6. Per-Expert Breakdown

How does each expert perform on its cluster?

In [ ]:
if expert_results:
    print(f'{"Cluster":<12} {"Bin jobs":>8} {"Scored":>8} {"MAE":>10} {"RMSE":>10} {"RT median":>10}')
    print('-' * 65)
    for label in BIN_LABELS:
        if label not in expert_results:
            continue
        p = expert_results[label]
        n_bin = len(bin_rows[label])
        rts = [r['runtime_seconds'] for r in bin_rows[label] if r.get('runtime_seconds')]
        rt_med = np.median(rts) if rts else 0
        print(f'{label:<12} {n_bin:>8,} {p["summary"]["rows_scored"]:>8,} '
              f'{p["mae"]:>10,.1f}s {p["rmse"]:>10,.1f}s {rt_med:>10,.0f}s')
else:
    print('No expert results available.')

## 7. Conclusions

In [ ]:
print('CONCLUSIONS')
print('=' * 60)
print(f'\nMAE difference (experts vs single): {mae_diff:+.1f}%')
print()

if mae_diff < -5:
    print('The per-bin expert approach shows meaningful improvement.')
    print('Next steps:')
    print('  - Test on more datasets (Eagle, Fugaku, Lassen)')
    print('  - Combine with log_target (issue #123) for further gains')
    print('  - Build the routing framework into the benchmark system')
elif mae_diff > 5:
    print('The single model outperforms per-bin experts.')
    print('XGBoost already captures wallclock patterns through tree splits.')
    print('Splitting the data just reduces training data per expert.')
    print('The mixture-of-experts approach may not be worth pursuing.')
else:
    print('Results are similar — the single model already captures most')
    print('of the wallclock signal through its internal tree splits.')
    print('The complexity of maintaining separate experts may not be justified.')
    print()
    print('However, this is a small-scale test. Consider:')
    print('  - Running on more data (full window instead of 30K sample)')
    print('  - Using different model types per bin (not just XGBoost everywhere)')
    print('  - Combining with log_target (issue #123)')

print()
print('Note: the current model uses post-hoc features (issue #122) which')
print('may mask the effect of wallclock-based routing. Re-evaluate after')
print('the feature allowlist fix is applied.')